# Publish Atlas-5K 10M (#58) and 25M (#59) to GitHub (ParadoxAI Lab)

No GPU needed: **Runtime → Change runtime type → CPU**, then **Run all**.

Before running, once: click the **key icon** (left sidebar) → **Add new secret** → name `GITHUB_TOKEN`, value = a GitHub
fine-grained token with *Contents: read and write* on `PARADOXAIlabs/paradox-world-model` → turn on **Notebook access**.
The token is read from Colab's secret store, never printed, and not saved anywhere.


In [ ]:
# 1. Drive and code
import os, shutil, glob, subprocess, base64
from google.colab import drive, userdata
drive.mount('/content/drive', timeout_ms=300000)
W = '/content/drive/MyDrive/PWM-10M'
!rm -rf /content/pwm && git clone -q -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1


In [ ]:
# 2. Gather the model files and run summaries from Drive, and the head-to-head results
I = '/content/rel_in'; H = f'{W}/small/h2h'; os.makedirs(I, exist_ok=True)
for f in ['w5k_atlas_10m_h24_g1_s600.pt', 'w5k_atlas_25m_h24_g2_s600.pt', 'w5k_atlas_10m_h24_g1_s600.json',
          'w5k_atlas_25m_h24_g2_s600.json', 'w5k_atlas_1m_h24_v2c_final_s600.json', 'log_w5k_atlas_25m_h24_g2.txt']:
    if os.path.exists(f'{W}/out/{f}'): shutil.copy(f'{W}/out/{f}', I)
J = '/content/rel_h2h'; os.makedirs(J, exist_ok=True)
for f in glob.glob(f'{H}/h2h_*.json') + glob.glob(f'{H}/paired_*.json'): shutil.copy(f, J)
!ls -la {I} {J} | head -40


In [ ]:
# 3. Build series files #58 and #59 (results inside), update #57, SERIES.json, SHA256SUMS, README
%cd /content/pwm
!python atlas/delhi/tools/release_series.py --in {I} --h2h {J} --repo /content/pwm
!git status --short | head -20


In [ ]:
# 4. Commit and push with the token from Colab's secret store (never printed)
tok = userdata.get('GITHUB_TOKEN')
auth = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
def git(*a):
    r = subprocess.run(['git', '-c', f'http.extraHeader={auth}', *a], cwd='/content/pwm', capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(tok, '***')); return r.returncode
subprocess.run(['git', 'add', 'models/series', 'atlas/results/colab'], cwd='/content/pwm')
subprocess.run(['git', '-c', 'user.name=Aditya Kumar', '-c', 'user.email=contact.paradoxailab@gmail.com', 'commit', '-q', '-m',
                'Atlas-5K 10M (#58) and 25M (#59) released: weights, dataset, metadata, validation, 2023 test and head-to-head inside; '
                'v2c 2023 test added to #57\n\nCo-Authored-By: Claude Opus 5.5 <noreply@anthropic.com>\n'
                'Claude-Session: https://claude.ai/code/session_01TCvEAwgYb9HyrCaEGd5YWa'], cwd='/content/pwm')
print('push exit code', git('push', 'origin', 'HEAD:atlas-v0.1'))
!git log --oneline -2
